# Clase 5 — La forma de los datos manda (I): imágenes

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** La arquitectura de una red **es una teoría sobre
> cómo está estructurado el mundo**. Una CNN afirma dos cosas: que lo que importa
> es *local*, y que un patrón *vale igual en cualquier posición*. Cuando eso es
> cierto, la red necesita muchísimos menos datos. Cuando es falso, te miente con
> mucha seguridad.

---

**Cuando termines este notebook vas a poder:**

1. Mostrar, con un experimento propio, que una red densa **nunca supo** que sus
   entradas formaban una grilla.
2. Escribir una convolución a mano con dos `for`, y después reemplazarla por
   `nn.Conv2d` habiendo verificado que dan **exactamente lo mismo**.
3. Entrenar una CNN chica, mirar los filtros que aprendió sola y compararla
   contra la red densa de la clase 4.
4. Hacer análisis de errores sobre un clasificador de imágenes y decir **qué
   está midiendo realmente**.

**Lo que hace falta traer:** la clase 4 (el training loop, train/val/test).

**Cuánto tarda.** Todo el notebook corre en CPU en **menos de 4 minutos**. Lo más
lento son los cuatro entrenamientos de la sección 2 (unos 70 segundos entre
todos). No hace falta GPU.

**Cómo se usa.** Corré las celdas en orden (`Shift + Enter`). Donde diga `# TODO`
te toca a vos. Los chequeos de este notebook **nunca explotan**: te dicen en
castellano qué salió y qué probar. Si algo se rompe de verdad, mirá
`recursos/setup.md`.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo y deja todo a mano; en tu máquina no hace prácticamente nada porque ya
tenés los archivos. Corrémela y seguí de largo: no hay nada que entender acá.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys

REPO = "https://github.com/johannperez/dlss"
RAMA = "claude/dl-course-social-scientists-279dd2"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                        REPO, CARPETA], check=True)
    raiz = buscar_raiz(CARPETA)

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab: revisá que la celda haya podido clonar (necesita internet).\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`."
    )

# Dependencias: en Colab casi todas vienen puestas. Instalamos sólo lo que falte.
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        print(f"Instalando {paquete}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete],
                       check=True)

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Imports, semilla y un ayudante

`ssdl` es el paquete del curso: la plomería que no queremos repetir en diez
notebooks. No esconde nada importante — está en `_shared/ssdl/` y se puede leer.

Fijamos la semilla en 1983 **siempre**: no hace al modelo mejor, hace a tu
resultado reproducible.

La función `revisar` de abajo es el corrector amable de este notebook. **No
explota nunca**: te dice si salió y, si no salió, qué mirar.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from ssdl import graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso
utils.resumen_del_entorno()


def revisar(condicion, bien, pista):
    """El corrector amable: nunca levanta una excepción, siempre dice algo útil."""
    if condicion:
        print("OK ·", bien)
    else:
        print("Todavía no ·", pista)
    return bool(condicion)

---

## 1. Las imágenes con las que vamos a jugar

> ### Un cartel grande, para que no haya malentendidos
>
> Las imágenes de esta primera mitad son **FashionMNIST**: 70.000 fotitos de
> 28×28 píxeles de prendas de ropa (Zalando Research, licencia MIT). Es un
> **sandbox técnico**, no es un dato de ciencias sociales, y no vamos a sacar
> ninguna conclusión sustantiva de acá. Sirve porque es chiquito, corre en CPU
> y todo el mundo puede mirar una imagen y decir si el modelo se equivocó.
>
> **En la sección 6 volvemos al dominio** con un archivo fotográfico y una
> pregunta de investigación de verdad.

Si no hay internet y tampoco están los archivos cacheados, la celda arma sola
un juego de figuras sintéticas y todo lo demás funciona igual. Esa es la regla
de la casa: **un notebook del curso nunca depende de la red**.

In [ ]:
import os
import pathlib
import sys

# datos/descargar.py sabe traer FashionMNIST (y usar la cache si ya está)
RAIZ = pathlib.Path(os.environ["DLSS_RAIZ"])
sys.path.insert(0, str(RAIZ / "datos"))


def figuras_sinteticas(n, semilla=0):
    """Plan B sin red: cuatro clases de figuras de 28x28, hechas con numpy."""
    rng = np.random.default_rng(semilla)
    X = np.zeros((n, 28, 28), dtype=np.uint8)
    y = rng.integers(0, 4, size=n).astype(np.int64)
    for i in range(n):
        f, c = rng.integers(4, 12), rng.integers(4, 12)
        alto, ancho = rng.integers(9, 16), rng.integers(9, 16)
        img = np.zeros((28, 28))
        if y[i] == 0:                                    # barras verticales
            img[f:f + alto, c:c + ancho:3] = 1.0
        elif y[i] == 1:                                  # barras horizontales
            img[f:f + alto:3, c:c + ancho] = 1.0
        elif y[i] == 2:                                  # bloque lleno
            img[f:f + alto, c:c + ancho] = 1.0
        else:                                            # marco hueco
            img[f:f + alto, c:c + ancho] = 1.0
            img[f + 2:f + alto - 2, c + 2:c + ancho - 2] = 0.0
        X[i] = np.clip(img * 200 + rng.normal(0, 12, (28, 28)) + 20, 0, 255)
    return X, y


try:
    import descargar
    _tr, _te = descargar.fashion_mnist("train"), descargar.fashion_mnist("t10k")
    if _tr is None or _te is None:
        raise RuntimeError("sin datos")
    (X_train_todo, y_train_todo), (X_test_todo, y_test_todo) = _tr, _te
    NOMBRES = descargar.ETIQUETAS_FASHION
    FUENTE = "FashionMNIST"
except Exception as e:                                   # noqa: BLE001
    print(f"[!] FashionMNIST no disponible ({type(e).__name__}). Voy al plan B sintético.")
    X_train_todo, y_train_todo = figuras_sinteticas(12000, 1)
    X_test_todo, y_test_todo = figuras_sinteticas(4000, 2)
    NOMBRES = ["barras verticales", "barras horizontales", "bloque", "marco"]
    FUENTE = "figuras sintéticas"

print(f"\nFuente: {FUENTE}")
print(f"forma del conjunto de entrenamiento: {np.shape(X_train_todo)}")
print(f"o sea: {np.shape(X_train_todo)[0]} imágenes de "
      f"{np.shape(X_train_todo)[1]}x{np.shape(X_train_todo)[2]} píxeles, en escala de grises")

### Miralas

Antes de escribir una línea de modelo, mirá el dato. Es la primera regla de
cualquier análisis y con imágenes es literal.

In [ ]:
idx = np.arange(16)
fig, _ = g.mostrar_imagenes(
    X_train_todo[idx],
    etiquetas=[NOMBRES[int(y_train_todo[k])] for k in idx],
    ncols=8, titulo=f"16 imágenes de {FUENTE} (sandbox técnico)")
plt.show()

### Una imagen es una tabla de números

Esto es lo único que ve la computadora: una grilla de números del 0 al 255.
Recortamos un pedacito de 8×8 y lo imprimimos al lado de la imagen, para que se
vea que el dibujo **es** la tabla.

> **Una aclaración sobre cómo se dibujan.** Acá matplotlib las muestra "al
> derecho": 0 es negro y 255 es blanco, que es como están guardadas en el
> archivo. En las slides y en las visualizaciones del curso las mismas imágenes
> se dibujan como **tinta sobre papel** (más número, más oscuro), porque así se
> leen mejor en un proyector. Es la misma tabla, dibujada con otro criterio —y
> es un buen recordatorio de que **la representación siempre es una decisión de
> quien muestra**, no una propiedad del dato.

In [ ]:
foto = X_train_todo[0]
recorte = foto[10:18, 8:16]        # ocho filas por ocho columnas del medio

fig, ejes = g.figura(11, 4.2, filas=1, columnas=2,
                     titulo="La misma cosa, dos veces: el dibujo y la tabla")
ejes[0].imshow(foto, cmap="gray")
ejes[0].add_patch(plt.Rectangle((7.5, 9.5), 8, 8, fill=False,
                                edgecolor=g.ACENTO, linewidth=2))
ejes[0].set_xticks([]); ejes[0].set_yticks([]); ejes[0].grid(False)
ejes[0].set_title(f"la foto entera: {foto.shape[0]}x{foto.shape[1]}")

ejes[1].imshow(recorte, cmap="gray")
for f in range(8):
    for c in range(8):
        v = int(recorte[f, c])
        ejes[1].text(c, f, str(v), ha="center", va="center", fontsize=7,
                     color="black" if v > 128 else "white")
ejes[1].set_xticks([]); ejes[1].set_yticks([]); ejes[1].grid(False)
ejes[1].set_title("el recorte de 8x8, con sus números")
plt.show()

print("El valor del píxel de la fila 12, columna 10:", int(foto[12, 10]))

---

## 2. El experimento del barajado

> ### Apostá primero
>
> Vamos a hacer lo siguiente: elegir **una** permutación de los 784 píxeles y
> aplicársela **a todas las imágenes, siempre la misma**. Después reentrenamos
> desde cero con esas imágenes revueltas.
>
> **Antes de correr nada, escribí tu apuesta:** ¿la red densa de la clase 4 va a
> andar peor? ¿Mucho peor? ¿Y una CNN?
>
> Sin la predicción fallida, el resultado es sólo un número más.

Los dos modelos son estos. Fijate que la red densa es **exactamente** la de la
clase 4: aplanar y dos capas.

In [ ]:
def red_densa(n_clases=10):
    """La red de la clase 4, tal cual: aplanar la imagen y dos capas."""
    return nn.Sequential(
        nn.Flatten(),                       # 28x28  ->  784 números en fila
        nn.Linear(28 * 28, 64), nn.ReLU(),
        nn.Linear(64, n_clases),
    )


def red_convolucional(n_clases=10):
    """Mira de a 3x3 tres veces; al final se queda con el máximo de cada mapa
    ('¿apareció esta pieza en ALGÚN lado?') y recién ahí decide."""
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
        nn.AdaptiveMaxPool2d(1), nn.Flatten(),
        nn.Linear(64, n_clases),            # la densa que DECIDE, al final
    )


print("perillas de la densa:", utils.contar_parametros(red_densa()))
print("perillas de la CNN:  ", utils.contar_parametros(red_convolucional()))

El loop de entrenamiento es el de la clase 4, con las mismas cinco líneas:
predecir, comparar, borrar, anotar, mover. Nada nuevo.

In [ ]:
SEMILLA = 1983
N_TRAIN, N_TEST, EPOCAS, LR, LOTE = 12000, 4000, 12, 2e-3, 64

rng = np.random.default_rng(SEMILLA)
itr = rng.choice(len(X_train_todo), size=min(N_TRAIN, len(X_train_todo)), replace=False)
ite = rng.choice(len(X_test_todo), size=min(N_TEST, len(X_test_todo)), replace=False)

# (batch, canales, alto, ancho) — "cuántas fotos, cuántas capas de color,
# cuán alta, cuán ancha". Esta cuaterna la vas a leer en todos los errores.
X_tr = torch.tensor(X_train_todo[itr] / 255.0, dtype=torch.float32).unsqueeze(1)
y_tr = torch.tensor(np.asarray(y_train_todo)[itr], dtype=torch.long)
X_te = torch.tensor(X_test_todo[ite] / 255.0, dtype=torch.float32).unsqueeze(1)
y_te = torch.tensor(np.asarray(y_test_todo)[ite], dtype=torch.long)
N_CLASES = int(max(y_tr.max().item(), y_te.max().item())) + 1

print("forma de X_tr:", tuple(X_tr.shape), " <- (batch, canales, alto, ancho)")


def entrenar(modelo, Xtr, ytr, Xte, yte, epocas=EPOCAS, lr=LR, lote=LOTE):
    """El loop de la clase 4. Devuelve la exactitud sobre lo que nunca vio."""
    torch.manual_seed(SEMILLA)
    perdida = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    gen = torch.Generator().manual_seed(SEMILLA)
    for _ in range(epocas):
        orden = torch.randperm(len(Xtr), generator=gen)
        for i in range(0, len(Xtr), lote):
            k = orden[i:i + lote]
            pred = modelo(Xtr[k])                  # 1. PREDECIR
            l = perdida(pred, ytr[k])              # 2. COMPARAR
            opt.zero_grad()                        # 3. BORRAR
            l.backward()                           # 4. ANOTAR
            opt.step()                             # 5. MOVER
    modelo.eval()
    with torch.no_grad():
        return (modelo(Xte).argmax(1) == yte).float().mean().item() * 100

### El barajado: una sola permutación, la misma para todas

Ojo con esto, porque es donde se juega el argumento: **no** revolvemos cada
imagen distinto. Elegimos una permutación de las 784 posiciones y se la
aplicamos a todas, siempre igual. No se pierde ni un número: sólo cambian de
casilla.

In [ ]:
permutacion = torch.tensor(np.random.default_rng(SEMILLA).permutation(28 * 28),
                           dtype=torch.long)


def barajar(X):
    """La misma permutación para todas las imágenes. Cero información perdida."""
    return X.reshape(len(X), 1, -1)[:, :, permutacion].reshape(len(X), 1, 28, 28)


X_tr_b, X_te_b = barajar(X_tr), barajar(X_te)

# la prueba de que no se perdió nada: los mismos números, en otro orden
originales = np.sort(X_te[0].numpy().ravel())
barajados = np.sort(X_te_b[0].numpy().ravel())
_ = revisar(np.allclose(originales, barajados),
        "la foto barajada tiene exactamente los mismos 784 números que la original",
        "algo se perdió en el camino: revisá la función barajar")

fig, _ = g.mostrar_imagenes(
    torch.cat([X_te[:4], X_te_b[:4]]).squeeze(1).numpy(), ncols=4,
    etiquetas=["original"] * 4 + ["barajada"] * 4,
    titulo="Arriba lo que ves vos. Abajo, lo mismo con los píxeles mudados de casilla.")
plt.show()

### Ahora sí: los cuatro entrenamientos

Cuatro modelos desde cero: densa y CNN, cada uno con las fotos normales y con
las fotos barajadas. Tarda alrededor de **70 segundos** en total. Aprovechá
para mirar de nuevo la figura de arriba: ¿vos podrías clasificar la fila de
abajo?

In [ ]:
import time

resultados = {}
for nombre, constructor, Xa, Xb in [
        ("densa · normal",   red_densa,          X_tr,   X_te),
        ("densa · barajada", red_densa,          X_tr_b, X_te_b),
        ("CNN · normal",     red_convolucional,  X_tr,   X_te),
        ("CNN · barajada",   red_convolucional,  X_tr_b, X_te_b)]:
    torch.manual_seed(SEMILLA)
    modelo = constructor(N_CLASES)
    t0 = time.time()
    acc = entrenar(modelo, Xa, y_tr, Xb, y_te)
    resultados[nombre] = {"exactitud": round(acc, 1), "modelo": modelo,
                          "parametros": utils.contar_parametros(modelo)}
    perillas = f"{resultados[nombre]['parametros']:,}".replace(",", ".")
    print(f"{nombre:18s} {acc:5.1f} %   ({perillas} perillas, {time.time()-t0:.0f} s)")

Y ahora la tabla que importa: **cuánto perdió cada uno al barajar**.

In [ ]:
caida_densa = resultados["densa · normal"]["exactitud"] - resultados["densa · barajada"]["exactitud"]
caida_cnn = resultados["CNN · normal"]["exactitud"] - resultados["CNN · barajada"]["exactitud"]

print(f"la red densa perdió {caida_densa:+.1f} puntos al barajar los píxeles")
print(f"la CNN         perdió {caida_cnn:+.1f} puntos al barajar los píxeles")
print()
print("La densa ni se enteró. La CNN sí. Esa diferencia es TODO lo que la CNN")
print("estaba sacando de saber quién estaba al lado de quién.")

fig, ax = g.figura(8.5, 4, titulo="Exactitud sobre lo que el modelo nunca vio",
                   ylabel="% de aciertos")
etiquetas = list(resultados)
valores = [resultados[k]["exactitud"] for k in etiquetas]
colores = [g.DATOS, g.DATOS, g.MODELO, g.MODELO]
barras = ax.bar(etiquetas, valores, color=colores)
for b, transp in zip(barras, [1, .45, 1, .45]):
    b.set_alpha(transp)                 # translúcido = con los píxeles barajados
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width() / 2, v + .6, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_ylim(0, 100)
plt.show()

### `TODO 1` — decilo con tus palabras

Escribí **en una oración** qué demuestra este experimento. No copies "la CNN es
mejor": eso no es lo que muestra.

*Pista: pensá qué información tenía la imagen original que la barajada no
tiene, y preguntate si el modelo la estaba usando. Si un modelo saca lo mismo
con y sin esa información, ¿la estaba usando?*

Escribila abajo y corré la celda: se guarda en el notebook y la vas a necesitar
para el cuestionario.

In [ ]:
# TODO 1: completá la frase entre comillas con tus palabras.
mi_conclusion = "..."

print(mi_conclusion)
_ = revisar(len(mi_conclusion) > 40 and mi_conclusion != "...",
        "quedó escrita: eso es lo que le vas a decir a alguien que te pregunte por qué usar una CNN",
        "todavía está el '...' — escribí tu oración, aunque no te convenza del todo. "
        "Una pista más: la palabra que buscás empieza con 'grilla'.")

> **Ojo con la lectura fácil.** La CNN también pierde *poco* (unos pocos
> puntos), y eso es honesto decirlo: FashionMNIST es un problema fácil y, aun
> con los píxeles revueltos, una CNN sigue siendo una máquina de combinar
> grupitos de nueve números con un clasificador arriba. Lo contundente no es
> *cuánto* pierde la CNN: es que la densa **no pierde nada**.
>
> La prueba más fuerte de que la CNN usa la grilla es la que sigue.

### La foto corrida: el gancho de la clase, medido

Acordate de la pregunta con la que abrimos: *la misma foto, corrida unos
píxeles a la derecha*. Para vos es la misma foto. Ahora lo medimos.

**Nadie reentrena nada**: son los dos modelos que ya entrenamos, mirando la
misma foto un poco más a la derecha.

In [ ]:
def correr(X, d):
    """La misma imagen, corrida d píxeles a la derecha. Por la izquierda entra
    fondo, no el otro borde: una foto corrida no da la vuelta."""
    if d == 0:
        return X
    Y = torch.zeros_like(X)
    Y[:, :, :, d:] = X[:, :, :, :-d]
    return Y


desplazamientos = [0, 1, 2, 3, 4, 6]
corrimiento = {}
for clave, nombre in [("densa", "densa · normal"), ("cnn", "CNN · normal")]:
    modelo = resultados[nombre]["modelo"]
    modelo.eval()
    fila = []
    for d in desplazamientos:
        with torch.no_grad():
            fila.append(round((modelo(correr(X_te, d)).argmax(1) == y_te)
                              .float().mean().item() * 100, 1))
    corrimiento[clave] = fila
    print(f"{clave:6s} " + "  ".join(f"{d}px:{v:5.1f}%"
                                     for d, v in zip(desplazamientos, fila)))

g.mostrar_imagenes(torch.cat([X_te[:1], correr(X_te[:1], 3)]).squeeze(1).numpy(),
                   etiquetas=["original", "corrida 3 píxeles"], ncols=2,
                   titulo="Para vos son la misma foto")
plt.show()

fig, ax = g.figura(8.5, 4.2, titulo="Qué pasa cuando la foto se corre unos píxeles",
                   xlabel="píxeles de corrimiento", ylabel="% de aciertos")
ax.plot(desplazamientos, corrimiento["densa"], "o-", color=g.DATOS,
        label="red densa (la de la clase 4)")
ax.plot(desplazamientos, corrimiento["cnn"], "o-", color=g.MODELO, label="CNN")
ax.set_ylim(0, 100)
ax.legend()
plt.show()

> **Esto es lo que compramos con el supuesto.** La densa se cae por el
> precipicio: para ella, correr la foto tres píxeles cambió **todas** las
> variables de entrada al mismo tiempo. La CNN aguanta mucho mejor, porque su
> estructura afirma que *un patrón vale igual en cualquier posición*.
>
> Y ojo con la contracara: la CNN también termina cayendo. La invariancia no es
> mágica ni infinita — es un supuesto que se cumple *hasta cierto punto*.

---

## 3. Filtros a mano: la convolución sin misterio

Una convolución es esto y nada más: una ventanita de 3×3 que recorre la imagen;
en cada posición multiplica lo que ve por nueve números y suma. Ese resultado es
un píxel del **mapa de activación**, que es un dibujo nuevo.

**Vocabulario firme, porque se confunde todos los años:**

| palabra | qué es |
|---|---|
| **filtro** (o *kernel*) | los nueve números. El pincel. |
| **mapa de activación** | el dibujo que sale. El cuadro pintado con ese pincel. |

Escribimos la convolución con dos `for`, a propósito lenta y transparente. Es la
misma cuenta que hace la viz `kernel-deslizante.html` y la misma que hace
PyTorch.

In [ ]:
def convolucion(imagen, filtro):
    """La convolución con dos for anidados. Lenta a propósito: se ve la cuenta.

    imagen: (alto, ancho) en numpy · filtro: (3, 3)
    devuelve: (alto-2, ancho-2)
    """
    alto, ancho = imagen.shape
    kf, kc = filtro.shape
    salida = np.zeros((alto - kf + 1, ancho - kc + 1))
    for f in range(salida.shape[0]):
        for c in range(salida.shape[1]):
            parche = imagen[f:f + kf, c:c + kc]     # lo que ve la ventana
            salida[f, c] = np.sum(parche * filtro)  # multiplicar y sumar
    return salida


foto_gris = X_test_todo[0] / 255.0

BORDE_VERTICAL = np.array([[1, 0, -1],
                           [2, 0, -2],
                           [1, 0, -1]], dtype=float)
BORDE_HORIZONTAL = BORDE_VERTICAL.T
DESENFOQUE = np.ones((3, 3)) / 9.0

mapas = {"borde vertical": convolucion(foto_gris, BORDE_VERTICAL),
         "borde horizontal": convolucion(foto_gris, BORDE_HORIZONTAL),
         "desenfoque": convolucion(foto_gris, DESENFOQUE)}

fig, ejes = g.figura(13, 3.6, filas=1, columnas=4,
                     titulo="La misma foto, pintada con tres pinceles distintos")
ejes[0].imshow(foto_gris, cmap="gray"); ejes[0].set_title("la foto")
for ax, (nombre, mapa) in zip(ejes[1:], mapas.items()):
    lim = np.abs(mapa).max()
    ax.imshow(mapa, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(nombre)
for ax in ejes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.show()

print("forma de la foto: ", foto_gris.shape)
print("forma del mapa:   ", mapas["borde vertical"].shape,
      " <- pierde un píxel de cada lado: la ventana no puede asomarse afuera")

### `TODO 2` — inventá un detector de diagonales

Mirá el filtro de bordes verticales: tiene los positivos de un lado y los
negativos del otro, y por eso se enciende donde hay un cambio **de izquierda a
derecha**.

*Pista 1: un detector de diagonales tiene que tener la zona positiva de un lado
de la diagonal y la negativa del otro.*
*Pista 2: probá poner los positivos en la esquina de arriba a la izquierda y los
negativos en la de abajo a la derecha.*
*Pista 3: la suma de los nueve números conviene que dé cero, si no el filtro
también cambia el brillo general.*

In [ ]:
# TODO 2: cambiá estos nueve números para que detecte diagonales.
mi_filtro = np.array([[0, 0, 0],
                      [0, 0, 0],
                      [0, 0, 0]], dtype=float)

mapa_mio = convolucion(foto_gris, mi_filtro)

fig, ejes = g.figura(9, 3.8, filas=1, columnas=3, titulo="Tu filtro en acción")
ejes[0].imshow(foto_gris, cmap="gray"); ejes[0].set_title("la foto")
lim = np.abs(mi_filtro).max() or 1
ejes[1].imshow(mi_filtro, cmap="RdBu_r", vmin=-lim, vmax=lim)
ejes[1].set_title("tus nueve números")
for f in range(3):
    for c in range(3):
        ejes[1].text(c, f, f"{mi_filtro[f, c]:g}", ha="center", va="center", fontsize=11)
lim2 = np.abs(mapa_mio).max() or 1
ejes[2].imshow(mapa_mio, cmap="RdBu_r", vmin=-lim2, vmax=lim2)
ejes[2].set_title("el mapa de activación")
for ax in ejes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.show()

# --- el chequeo: se lo probamos contra dos imágenes hechas a propósito -------
diagonal = np.eye(9)                        # una raya de arriba-izq a abajo-der
antidiagonal = np.fliplr(np.eye(9))         # la raya al revés
r_diag = np.abs(convolucion(diagonal, mi_filtro)).sum()
r_anti = np.abs(convolucion(antidiagonal, mi_filtro)).sum()

_ = revisar(np.abs(mi_filtro).sum() > 0,
        "el filtro ya no es todo ceros",
        "sigue en todo ceros: todo por cero es cero y el mapa sale vacío")
_ = revisar(abs(mi_filtro.sum()) < 1e-9,
        "los nueve números suman cero: no te cambia el brillo general",
        f"suman {mi_filtro.sum():g}. No está mal, pero además de detectar bordes "
        "te aclara o te oscurece la imagen entera")
_ = revisar(r_diag > 1.35 * r_anti or r_anti > 1.35 * r_diag,
        "distingue una diagonal de la otra: eso es un detector de diagonales",
        f"reacciona casi igual a las dos diagonales ({r_diag:.1f} vs {r_anti:.1f}). "
        "Probá con los positivos arriba a la izquierda y los negativos abajo a la derecha")

### `TODO 3` — el filtro que no hace nada

Un ejercicio de un minuto que consolida la mecánica.

*Pista: si en cada posición querés que el resultado sea el píxel del centro y
nada más, ¿cuánto tiene que valer cada uno de los nueve números? Pensá en la
multiplicación por 1 y en la multiplicación por 0.*

In [ ]:
# TODO 3: escribí el filtro que deja la imagen igualita (la "identidad").
filtro_identidad = np.array([[0, 0, 0],
                             [0, 0, 0],
                             [0, 0, 0]], dtype=float)

salida = convolucion(foto_gris, filtro_identidad)
centro = foto_gris[1:-1, 1:-1]

_ = revisar(np.allclose(salida, centro),
        "sale la imagen igual (menos el borde de un píxel, que la ventana no alcanza)",
        "todavía no devuelve la imagen. Acordate: querés quedarte SÓLO con el "
        "píxel del centro de la ventana, así que ocho de los nueve números tienen "
        "que anular lo que ven")

---

## 4. `nn.Conv2d`: lo mismo, pero rápido

Regla de la casa: **ninguna abstracción antes de haber hecho a mano lo que
esconde**. Ya escribiste la convolución con dos `for`; ahora la reemplazamos por
la de PyTorch, y **verificamos que dan el mismo número**. Si no diera lo mismo,
sería que una de las dos está mal.

In [ ]:
# la misma cuenta, ahora en torch. OJO con las dimensiones:
#   la imagen entra como (batch, canales, alto, ancho)
#   el filtro entra como (filtros_de_salida, canales_de_entrada, alto, ancho)
capa = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    capa.weight[:] = torch.tensor(BORDE_VERTICAL, dtype=torch.float32).reshape(1, 1, 3, 3)

entrada = torch.tensor(foto_gris, dtype=torch.float32).reshape(1, 1, 28, 28)
with torch.no_grad():
    salida_torch = capa(entrada)[0, 0].numpy()

salida_a_mano = convolucion(foto_gris, BORDE_VERTICAL)

print("a mano  ->", salida_a_mano.shape, " torch ->", salida_torch.shape)
print("diferencia máxima entre las dos:", float(np.abs(salida_a_mano - salida_torch).max()))
_ = revisar(np.allclose(salida_a_mano, salida_torch, atol=1e-5),
        "el doble for y nn.Conv2d dan exactamente lo mismo. No hay magia adentro",
        "dan distinto: revisá la forma del filtro (1, 1, 3, 3) y que sea el mismo filtro")

> **Leer un error de forma es contenido, no un accidente.** Si te olvidás el
> `unsqueeze` o le pasás una imagen de 28×28 pelada, PyTorch te dice algo como
> *"expected 4-dimensional input... but got 2-dimensional input"*. Ese mensaje
> te está diciendo la cuaterna que falta: **(batch, canales, alto, ancho)** —
> cuántas fotos, cuántas capas de color, cuán alta, cuán ancha. La mitad de los
> errores de esta clase se arreglan leyendo esa línea despacio.

### Ver los filtros aprendidos

*(Esta es la celda que promete la visualización `viz/jerarquia.html`: allá ves un
esquema ilustrativo; acá ves los filtros de verdad de nuestra propia CNN.)*

Los nueve números de cada filtro son **parámetros**: se aprenden con la misma
loss y el mismo descenso por gradiente de la clase 3. Nadie los escribió.

Miremos los 16 filtros de la primera capa de la CNN que entrenamos arriba.

In [ ]:
cnn = resultados["CNN · normal"]["modelo"]
pesos = cnn[0].weight.detach()          # (16, 1, 3, 3)

fig = g.grilla_de_filtros(pesos, ncols=8,
                          titulo="Los 16 filtros de la primera capa, aprendidos por descenso por gradiente")
plt.show()

# y los dibujos que producen sobre una foto concreta
with torch.no_grad():
    activaciones = torch.relu(cnn[0](X_te[:1]))[0]      # (16, 28, 28)

fig, _ = g.mostrar_imagenes(activaciones.numpy(), ncols=8, cmap="magma",
                            etiquetas=[f"filtro #{k}" for k in range(16)],
                            titulo="Los 16 mapas de activación de UNA foto: 16 dibujos distintos de lo mismo")
plt.show()

### `TODO 4` — mirá los filtros aprendidos

No hay código acá: hay que mirar. Compará la grilla de filtros de arriba con los
tres que hiciste a mano (bordes verticales, horizontales, desenfoque).

- ¿Ves alguno que tenga una mitad azul y una mitad roja, como el de bordes?
- ¿Ves alguno casi todo del mismo color, como el desenfoque?
- ¿Ves alguno que no se parezca a nada y que no sepas nombrar?

Ese último grupo es importante: **la mayoría de los filtros no tienen un nombre
en castellano**, y ese es exactamente el problema de interpretabilidad que
retomamos en la clase 10. La CNN es un ejército de codificadores con reglas
aprendidas... y las reglas no están escritas en ningún libro de códigos.

In [ ]:
# TODO 4: escribí qué viste (número de filtro y a qué te recordó).
mi_lectura_de_filtros = "..."

print(mi_lectura_de_filtros)
_ = revisar(mi_lectura_de_filtros != "..." and len(mi_lectura_de_filtros) > 20,
        "listo: acabás de hacer análisis cualitativo sobre parámetros de una red",
        "escribí aunque sea una línea. Si ninguno te recuerda a nada, eso TAMBIÉN "
        "es un hallazgo y conviene anotarlo")

---

## 5. Volvemos a ciencias sociales: un archivo fotográfico

Se terminó el sandbox. Ahora la situación real, la que aparece en una
investigación:

> Tenés un archivo fotográfico digitalizado — prensa gráfica, un fondo
> documental, fotos de campaña — y querés clasificar las imágenes en
> **"hay una movilización"** / **"no hay una movilización"**. Etiquetaste unas
> cuantas a mano y querés que el modelo haga el resto.

El corpus que vamos a usar es **sintético**: lo genera numpy con un proceso
generador que conocemos. Eso no es una limitación, es la gracia: **sabemos la
verdad**, y por eso podemos mostrar exactamente en qué se equivoca el modelo y
por qué. País ficticio, cero fotos de gente real.

Hay tres tipos de escena y dos etiquetas:

| escena | etiqueta | qué es |
|---|---|---|
| **multitud** | 1 | muchas cabecitas repartidas bajo el horizonte |
| **edificio** | 0 | un bloque grande con ventanas: una fachada, una tarima |
| **follaje** | 0 | racimos de manchitas: copas de árboles |

El **follaje está puesto a propósito**. Tiene la misma *textura* que una
multitud y ninguna persona adentro. Guardate esa carta: la vamos a cobrar en el
`TODO 5`.

In [ ]:
LADO = 32


def _suavizar(img):
    """Un desenfoque 3x3. Las fotos de archivo no tienen bordes de bisturí."""
    L = img.shape[0]
    k = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], float)
    k /= k.sum()
    p = np.pad(img, 1, mode="edge")
    out = np.zeros_like(img)
    for j in range(3):
        for i in range(3):
            out += k[j, i] * p[j:j + L, i:i + L]
    return out


def archivo_visual(n, semilla, lado=LADO, prop_follaje=0.34):
    """Archivo fotográfico SINTÉTICO. Devuelve (imágenes, etiquetas, tipo)."""
    rng = np.random.default_rng(semilla)
    X = np.zeros((n, lado, lado))
    y = np.zeros(n, dtype=np.int64)
    tipo = []
    yy, xx = np.mgrid[0:lado, 0:lado]
    for k in range(n):
        base = rng.uniform(140, 205)
        img = np.tile(np.linspace(base, base - 40, lado), (lado, 1)).T
        r = rng.random()
        if k % 2 == 0:                                    # ---- multitud
            y[k] = 1
            tipo.append("multitud")
            hor = rng.integers(3, 18)
            img[hor:, :] = rng.uniform(110, 175)
            for _ in range(rng.integers(45, 120)):
                cy, cx = rng.integers(hor, lado - 1), rng.integers(0, lado)
                img[(yy - cy) ** 2 + (xx - cx) ** 2
                    <= rng.uniform(0.8, 2.0) ** 2] = rng.uniform(20, 90)
        elif r < prop_follaje:                            # ---- follaje
            y[k] = 0
            tipo.append("follaje")
            hor = rng.integers(3, 18)
            img[hor:, :] = rng.uniform(110, 175)
            for _ in range(rng.integers(2, 6)):
                by, bx = rng.integers(2, lado - 4), rng.integers(2, lado - 4)
                for _ in range(rng.integers(15, 40)):
                    cy = int(np.clip(by + rng.normal(0, 3.5), 0, lado - 1))
                    cx = int(np.clip(bx + rng.normal(0, 3.5), 0, lado - 1))
                    img[(yy - cy) ** 2 + (xx - cx) ** 2
                        <= rng.uniform(0.8, 2.0) ** 2] = rng.uniform(20, 90)
        else:                                             # ---- edificio
            y[k] = 0
            tipo.append("edificio")
            a, an = rng.integers(10, 20), rng.integers(12, 24)
            f0 = lado - a - rng.integers(0, 5)
            c0 = rng.integers(0, lado - an)
            img[f0:f0 + a, c0:c0 + an] = rng.uniform(60, 110)
            for fv in range(f0 + 2, f0 + a - 2, 4):
                for cv in range(c0 + 2, c0 + an - 2, 5):
                    img[fv:fv + 2, cv:cv + 3] = rng.uniform(170, 215)
        X[k] = np.clip(_suavizar(img) + rng.normal(0, 7, (lado, lado)), 0, 255)
    return X.astype(np.uint8), y, np.array(tipo)


X_arch, y_arch, tipo_arch = archivo_visual(400, 11)      # lo que etiquetamos
X_val, y_val, tipo_val = archivo_visual(400, 22)         # lo que NUNCA vio

fig, _ = g.mostrar_imagenes(X_arch[:12], ncols=6,
                            etiquetas=[f"{t}\n(etiqueta {int(e)})"
                                       for t, e in zip(tipo_arch[:12], y_arch[:12])],
                            titulo="El archivo SINTÉTICO: 12 fotos de las 400 etiquetadas")
plt.show()

import collections
cuenta = collections.Counter(str(t) for t in tipo_arch)
print("qué hay en el archivo de entrenamiento:", dict(cuenta))

Entrenamos **la misma CNN** de antes. Fijate que no tocamos ni una línea de la
arquitectura, aunque ahora las imágenes son de 32×32 y no de 28×28: el
`AdaptiveMaxPool2d(1)` del final se acomoda solo. Eso también es un supuesto
hecho estructura.

In [ ]:
X_arch_t = torch.tensor(X_arch / 255.0, dtype=torch.float32).unsqueeze(1)
y_arch_t = torch.tensor(y_arch, dtype=torch.long)
X_val_t = torch.tensor(X_val / 255.0, dtype=torch.float32).unsqueeze(1)
y_val_t = torch.tensor(y_val, dtype=torch.long)

torch.manual_seed(SEMILLA)
modelo_archivo = red_convolucional(2)
acc_archivo = entrenar(modelo_archivo, X_arch_t, y_arch_t, X_val_t, y_val_t,
                       epocas=10, lote=32)
print(f"exactitud sobre las 400 fotos que nunca vio: {acc_archivo:.1f} %")

with torch.no_grad():
    prediccion = modelo_archivo(X_val_t).argmax(1).numpy()

g.matriz_de_confusion(y_val, prediccion,
                      etiquetas=["sin movilización", "hay movilización"],
                      nombre_positivo="hay movilización")
plt.show()

g.frase_de_metricas(y_val, prediccion, fenomeno="fotos con movilización")

> **La matriz de confusión aparece acá por primera vez y no la vamos a explicar
> en detalle: la clase 9 vive de esto.** Por ahora quedate con la regla de la
> casa: prohibido reportar una métrica sin poder decirla en voz alta en
> castellano. Eso es lo que imprime `frase_de_metricas`.

### `TODO 5 (criterio)` — el ejercicio más importante del notebook

Un 97 % no te dice **qué** aprendió el modelo. Los errores sí.

La celda de abajo grafica los casos en los que se equivocó. Mirálos un minuto,
en silencio, antes de leer nada más, y escribí qué tienen en común.

*Pista: el conocimiento está en los errores, no en los aciertos. Y una segunda:
fijate si los errores se reparten parejo entre los tres tipos de escena o si se
amontonan en uno.*

In [ ]:
errores = np.where(prediccion != y_val)[0]
print(f"{len(errores)} errores sobre {len(y_val)} fotos\n")

n_mostrar = min(9, len(errores))
if n_mostrar:
    corto = {0: "NO", 1: "SÍ"}          # SÍ = hay movilización
    fig, _ = g.mostrar_imagenes(
        X_val[errores[:n_mostrar]], ncols=3,
        etiquetas=[f"{corto[y_val[k]]} · {tipo_val[k]}" for k in errores[:n_mostrar]],
        predicciones=[f"dijo {corto[prediccion[k]]}" for k in errores[:n_mostrar]],
        titulo="Los errores  ·  SÍ = hay movilización, NO = no hay")
    plt.show()

# la tabla que hay que mirar: ¿los errores se reparten parejo?
print(f"{'escena':10s} {'cuántas':>8s} {'exactitud':>10s} {'errores':>8s}")
for t in ["multitud", "follaje", "edificio"]:
    sel = tipo_val == t
    print(f"{t:10s} {sel.sum():8d} {100*(prediccion[sel]==y_val[sel]).mean():9.1f}% "
          f"{int((prediccion[sel]!=y_val[sel]).sum()):8d}")

In [ ]:
# TODO 5: ¿qué tienen en común los errores? ¿Qué está midiendo REALMENTE
# el modelo cuando dice "hay movilización"?
que_mide_realmente = "..."

print(que_mide_realmente)
_ = revisar(que_mide_realmente != "..." and len(que_mide_realmente) > 30,
        "escrito. Ese párrafo es la sección de limitaciones de tu paper",
        "escribilo igual, aunque sea una hipótesis. Pista: mirá la columna "
        "'exactitud' por tipo de escena y preguntate qué tienen en común una "
        "multitud y una copa de árbol vistas desde lejos y en baja resolución")

> **Lo que acaba de pasar.** El modelo acierta muchísimo *en promedio*, y sin
> embargo los errores **no se reparten al azar**: se amontonan en el follaje.
> Traducido: el modelo no aprendió "hay gente". Aprendió *"hay muchas manchitas
> chiquitas repartidas"*, que en este archivo casi siempre coincide con gente.
>
> Eso tiene un nombre en ciencias sociales y ya lo conocés: es un problema de
> **validez de constructo**. El indicador correlaciona con lo que te interesa,
> pero no *es* lo que te interesa, y el día que la correlación se rompe —otro
> archivo, otra ciudad, otra época, más árboles— el modelo se rompe sin avisar.
>
> Fijate además que esto es la contracara exacta de la ventaja que celebramos en
> la sección 2: la CNN es invariante a la posición, así que le da igual que las
> manchitas estén arriba (copas) o abajo (cabezas). **El supuesto que te salva
> es el mismo que te hunde.**

---

## 6. Qué te llevás

1. **La arquitectura es una teoría sobre el mundo.** La CNN afirma que lo que
   importa es local y que un patrón vale igual en cualquier posición. Lo
   mediste: con los píxeles barajados la densa no perdió nada y la CNN sí; con
   la foto corrida tres píxeles la densa se desplomó y la CNN aguantó.
2. **Un filtro son nueve números y un mapa de activación es un dibujo.** Lo
   escribiste con dos `for` y después comprobaste que `nn.Conv2d` hace la misma
   cuenta. No hay magia adentro; lo único nuevo es la *forma* de la máquina.
3. **La exactitud global no te dice qué está midiendo el modelo. Los errores
   sí.** Y cuando los errores se concentran en un subgrupo, ahí está tu sección
   de limitaciones.

### `[EL LUNES]`

Si en tu investigación hay imágenes —archivo fotográfico, prensa gráfica,
material etnográfico visual, actas escaneadas— podés empezar esta semana con
tres cosas y **ninguna** requiere entrenar nada:

1. Etiquetá a mano 100 imágenes de cada categoría. Es una tarde, y sin eso no
   hay proyecto.
2. Definí **antes** la partición: si tus fotos vienen de distintos archivos,
   ciudades o épocas, partí por ahí y no al azar (clase 4).
3. Escribí en una línea cuál es tu **baseline**: ¿qué acierta el "siempre digo
   que no"? Si es 85 %, un modelo del 88 % no es una noticia.

### Las visualizaciones de esta clase

Están en `clases/05-imagenes-y-convoluciones/viz/` y abren con doble clic, sin
internet. Si algo de hoy quedó flojo, la que conviene volver a abrir es:

- `vecindad-de-pixeles.html` — el barajado y la foto corrida, con los números medidos.
- `kernel-deslizante.html` — los nueve números, editables, sobre cinco imágenes.
- `jerarquia.html` — de qué está hecha una «rueda».
- `supuestos-de-arquitectura.html` — **la que conviene tener a mano el día que
  elijas un modelo para tus propios datos.**

Y si vas a **leer** un paper que clasifica imágenes, las tres preguntas son:
¿qué está midiendo realmente?, ¿valen acá los supuestos de la arquitectura?,
¿qué pasa fuera de la región donde se entrenó?

### El puente a la clase 6

La estructura del espacio pide convoluciones. Pero el texto no es una grilla:
es una **secuencia**, y el significado de una palabra depende de palabras que
pueden estar muy lejos. *"La comisión rechazó al ministro"* y *"el ministro
rechazó a la comisión"* tienen exactamente las mismas palabras.

La clase que viene: cómo se convierte un discurso en números **sin perder el
orden**.